# 5. Train the transformer

Training runs in WSL on the GPU (about a day for the full run). From a WSL terminal:

    bash /mnt/d/GITHUB_SOFTWARE/SPTnano-core/wsl/train_transformer.sh --smoke_test
    bash /mnt/d/GITHUB_SOFTWARE/SPTnano-core/wsl/train_transformer.sh --out_dir /mnt/d/TRANSFORMER_DEVELOPMENT/saved_data/my_model

Two stages, settings in `transformer:` of `configs/pipeline.yaml`:

1. **Contrastive pretraining** (400 epochs, batch 1024): the encoder learns to give a window and its augmented
   copy the same 16-D embedding (InfoNCE, temperature 0.2; windows from the same track are not used as negatives).
2. **Classifier** (up to 200 epochs, batch 512): a small MLP head on the pretrained encoder, trained with
   cross-entropy on a class-balanced 10% of the labelled training windows.

Early stopping after 15 epochs without a better validation loss. Each stage saves `best_model.pt` and `final_model.pt`.

The paper model is already in `models/block3_long_large/`, so you only need this to retrain.

In [ ]:
import sys
sys.path.insert(0, "..")  # so `import sptnano` works from this folder

import polars as pl

from sptnano import config

paths = config.load_paths()
settings = config.load_settings()

import json

from sptnano import evaluate, model

In [ ]:
print(json.dumps(settings["transformer"], indent=2))
print(json.dumps(model.read_manifest(paths["model_dir"]), indent=2))

## Loss curves

Set `model_dir` to your own run (the `--out_dir` you trained into) to compare.

In [ ]:
model_dir = paths["model_dir"]
losses = evaluate.loss_table(model_dir)
evaluate.plot_loss_curves(losses)
losses.groupby("stage")["validation"].agg(["min", "idxmin", "count"])